# Fundamentals 00.4 - Runtime Scheduler API

**Checkpoint 1 materializado.** Antes de construir tools o agentes complejos, este notebook prueba la capa que protege el kernel: `toolkit.scheduler(...)`, `toolkit.runtime(...)`, `runtime=` en `toolkit.agent(...)` y la ruta explicita con `engine=...`.

Idea clave:

```text
Scheduler = limites de ejecucion
Runtime   = provider/backend + modelo + scheduler
Agent     = tools + runtime + contrato de ejecucion
```

## 0) Setup del repo

Esta celda permite ejecutar el notebook desde la raiz del repo o desde cualquier subcarpeta de `tutorials/notebooks`.

In [ ]:
import json
import time

import agentic_systems as toolkit

PRETTY = False


def show(obj, title: str | None = None) -> None:
    if title:
        print(f"\n=== {title} ===")
    print(json.dumps(obj, indent=2, ensure_ascii=False, default=str))


## Parametros de `RunPolicy`

`RunPolicy` declara como debe comportarse una ejecucion antes de llamar al agente o al runtime. No es metadata decorativa: limita loops, define reparacion, controla trazas y hace que el resultado sea evaluable.

| Parametro | Que controla | Uso recomendado |
|---|---|---|
| `max_turns` | Numero maximo de turnos internos del agente. | Mantenerlo bajo en notebooks para evitar loops largos. |
| `max_tool_calls` | Numero maximo de llamadas a tools. | Declararlo cuando el ejercicio espera tools concretas. |
| `max_tokens` | Limite de tokens del modelo cuando el provider lo soporta. | util en providers LM; puede quedar `None` en `python-runtime`. |
| `temperature` | Aleatoriedad del modelo. | `0.0` para tutoriales reproducibles; `None` delega al provider. |
| `tool_choice` | Estrategia de seleccion de tools, por ejemplo `auto`. | `auto` cuando el agente decide; explicito cuando quieres forzar una tool. |
| `repair` | Permite reparacion automatica de salidas o tool calls invalidas. | `True` para UX robusta; `False` si quieres ver fallos crudos. |
| `max_repairs` | Maximo de intentos de reparacion. | `1` o `2` en tutoriales para mostrar control sin ocultar errores. |
| `finalize` | Que hacer al agotar turnos, por ejemplo `on_max_turns`. | Mantenerlo explicito en agentes LM evaluables. |
| `trace` | Nivel de trazabilidad (`compact`, `debug`, etc.). | `compact` para notebooks; `debug` solo para diagnostico. |
| `strict` | Si el contrato debe aplicarse de forma estricta. | `True` para ensenar API y evitar ambiguedad. |


## Escenario didactico compartido

Todos los notebooks de `tutorials/` usan este mismo problema para que puedas comparar la API sin cambiar de caso cada vez:

```text
Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final
```



In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

# La estructura solicitada por el usuario se materializa como datos simples.
# No es un parser ni una respuesta precocinada: solo representa la seccion `Dime:`.
REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

toolkit.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Escenario didactico  visible")


## 1) Crear `SchedulerConfig`

El scheduler no ejecuta nada por si solo. Es una estructura declarativa con limites seguros para notebooks/sandbox.

In [ ]:
scheduler = toolkit.scheduler(
    timeout_s=30,
    max_retries=1,
    max_tool_calls=3,
    max_turns=4,
    max_concurrency=1,
    backoff_s=0.0,
)

show(scheduler.to_dict(), "SchedulerConfig")

## 2) Crear `RuntimeConfig`

El runtime responde: **donde corre** Aqui usamos `python-runtime` porque es reproducible sin AWS. Para Bedrock se usa el mismo patron cambiando el provider.

In [ ]:
runtime = toolkit.runtime(
    provider="python-runtime",
    model="python-runtime",
    region="local",
    scheduler=scheduler,
    metadata={"checkpoint": "1_runtime_scheduler"},
)

show(runtime.to_dict(), "RuntimeConfig")

## 3) Tool normal que siempre regresa diccionario

La convencion recomendada para intercambio entre piezas es: **entrada dict, salida dict**.

In [ ]:
@toolkit.tool
def duplicar(value: int) -> dict:
    """Duplica un numero."""
    return {"input": value, "result": value * 2}


agent = toolkit.agent(
    name="runtime_scheduler_agent",
    instructions="Usa la tool disponible y responde con datos estructurados.",
    tools=[duplicar],
    runtime=runtime,
)

result = agent.run({"tool": "duplicar", "input": {"value": 21}})
toolkit.human_result(result, pretty=PRETTY, title="Run con runtime=runtime")

## 4) Ver metadata del scheduler en el resultado

El resultado visible vive en `result.final`; la evidencia en `result.data`; la metadata de runtime/scheduler se queda en el envelope.

In [ ]:
show({
    "final": result.final,
    "data": result.data,
    "runtime_meta": result.meta.get("runtime"),
    "scheduler_meta": result.meta.get("scheduler"),
    "scheduler_execution": result.meta.get("scheduler_execution"),
    "usage_scheduler": result.usage.get("scheduler"),
}, "Separacion final/data/runtime")

## 5) Retry controlado

Este ejemplo falla una vez y luego se recupera. Sirve para probar que `max_retries` no es decorativo.

In [ ]:
attempts = {"count": 0}

@toolkit.tool
def flaky_increment(value: int) -> dict:
    """Falla una vez y luego suma uno."""
    attempts["count"] += 1
    if attempts["count"] == 1:
        raise RuntimeError("fallo controlado para probar retry")
    return {"input": value, "result": value + 1, "attempts": attempts["count"]}

retry_runtime = toolkit.runtime(
    provider="python-runtime",
    scheduler=toolkit.scheduler(timeout_s=30, max_retries=1, max_tool_calls=2, max_turns=3),
)
retry_agent = toolkit.agent(name="retry_agent", tools=[flaky_increment], runtime=retry_runtime)
retry_result = retry_agent.run({"tool": "flaky_increment", "input": {"value": 10}})

toolkit.human_result(retry_result, pretty=PRETTY, title="Run con retry")
show({"attempts": attempts, "scheduler_execution": retry_result.meta.get("scheduler_execution")})

## 6) Timeout controlado

Este bloque muestra el patron. El timeout esta configurado bajo para que el error sea rapido. En notebooks reales se usan limites mas altos.

In [ ]:
@toolkit.tool
def slow_tool(value: int) -> dict:
    """Tool lenta para probar timeout."""
    time.sleep(0.2)
    return {"value": value}

timeout_runtime = toolkit.runtime(
    provider="python-runtime",
    scheduler=toolkit.scheduler(timeout_s=0.05, max_retries=0, max_tool_calls=1, max_turns=1),
)
timeout_agent = toolkit.agent(name="timeout_agent", tools=[slow_tool], runtime=timeout_runtime)
timeout_result = timeout_agent.run({"tool": "slow_tool", "input": {"value": 1}})

toolkit.human_result(timeout_result, pretty=PRETTY, title="Run con timeout controlado")
show({"ok": timeout_result.ok, "errors": timeout_result.errors, "meta": timeout_result.meta})

## 7) Runtime explicito en agente

Este bloque repite la ejecucion con `agent(..., runtime=runtime)`. La idea es ensenar el camino canonico: el agente no decide backend ni limites; recibe un runtime ya configurado.

In [ ]:
runtime_agent = toolkit.agent(
    name="runtime_agent",
    tools=[duplicar],
    runtime=runtime,
)
runtime_result = runtime_agent.run({"tool": "duplicar", "input": {"value": 7}})

toolkit.human_result(runtime_result, pretty=PRETTY, title="Runtime configurado explicitamente")

## Lo importante

- `scheduler` protege ejecucion: timeout, retries, max tool calls, max turns.
- `runtime` selecciona provider/backend y carga el scheduler.
- `agent(..., runtime=runtime)` es la ruta recomendada para codigo nuevo.
- La salida visible esta en `result.final`; runtime/scheduler no contaminan esa respuesta.

## Coverage API de este notebook

Esta tabla deja explicito que parte de Agentic Systems queda materializada aqui.

In [ ]:
api_coverage = [
    {
        "api": "SchedulerConfig",
        "description": "Declara limites de ejecucion sin acoplar la logica al agente."
    },
    {
        "api": "RunPolicy budgets",
        "description": "Muestra los presupuestos de ejecucion que protegen la sesion."
    },
    {
        "api": "runtime=",
        "description": "Prueba la via recomendada para inyectar runtime en el agente."
    },
    {
        "api": "python-runtime smoke",
        "description": "Valida el backend local reproducible sin depender de AWS."
    },
    {
        "api": "shared scenario declared",
        "description": "Reutiliza el mismo problema aritmetico para comparar notebooks 1:1."
    }
]

toolkit.show({'notebook': '00_runtime_scheduler_api.ipynb', 'api_coverage': api_coverage})

## Simbolos API explicados

Este notebook se alinea con `docs/API.md` y ensena estos simbolos publicos:

- `toolkit.scheduler`: Declaracion de limites transversales.
- `SchedulerConfig`: Tipo publico detras del scheduler.
- `RuntimeConfig`: Runtime que recibe scheduler.
- `timeout_s / max_retries / max_turns / max_tool_calls`: Politica operativa que limita ejecucion.
- `toolkit.agent`: Agente que hereda runtime y scheduler.

